- Optimizing Nueral Network

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [3]:
torch.manual_seed(42)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Divice is {device}")

Divice is cuda


In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
dataset_path = "/content/drive/MyDrive/DataSets/fashion-mnist_train.csv"

In [7]:
df = pd.read_csv(dataset_path)

In [8]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
X = df.drop(columns="label")
y = df["label"]

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
X_train = X_train/255.0
X_test = X_test/255.0

In [12]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features.to_numpy(),dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(),dtype=torch.long)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [13]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [14]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

- **Dropout:**
    - Applied to the hidden layers
    - Applied after ReLU activation function
    - Randomly turns off p% neurons in the hidden layer during each forward pass (results simpify neural network and get different neural networks at each forward pass)
    - This has regularization effect (due to turn off of diffent neurons in each forward pass, we train the new neural network each time)
    - During evaluation dropout is not used


- **Batch Normalization:**
    - It improves training stability
    - Applied to Hidden Layers:
        - Typically applied to the hidden layers of neural network, but not to the output layer.
    - Applied After Linear Layer and Before Activation Functions:
        - Normalizes the output of the preciding layer (e.g., after nn.Linera) and is usually followed by an activation function (e.g., ReLU)
    - Normalizes Activations:
        - Compute the mean and variance of the activations within a mini-batch and uses these statistics to normalize the activations.
    - Includes Learnable Parameters:
        - Introduces two learnable parameters, gamma(scaling) abd beta(shifting), which allow the neural network to adjust the normalized outputs.
    - Improves Training Stability:
        - Reduce internal covariate shift, stabilizing the training process and allowing the use of higher learning rate.
    - Regularization Effect:
        - Introduces some regularization because the statistics are computed over a mini-batch, adding noise to training process.
    - Consistent During Evaluation:
        - During evaluation BatchNorm uses the running mean and variance accumulated during training, rather than recomputing them from mini-batch

- **Regularization (L2):**
    - Neural Networks are essentially optimazation problem, where we have loss function,(like in regression MSE), and parameters(weights and biases). Our works is to find out the value of weight and biases so we can find out minimum value of loss.
    - Applied to Model Weights:
        - Regularization is applied to the weights of the model to penalize large values and encourages smaller, more genralizable weights
    - Introduce via Functionsal or Optimizer:
        - Adds a penalty term $\lambda \sum w_i^2$ to the loss function in **L2 Regularization**.

            $$
            Loss_{reg} = Loss_{original} + \lambda \sum w_i^2
            $$

        - In **Weight Decay**, directly modifies the gradient update rule to include $\lambda w_i$, effectively shrinking weights during training.

            $$
            w \leftarrow w - \eta (\nabla Loss + \lambda w)
            $$
        
    - Penalizes Large Weights:
        - Encourages the network to distribute learning across multiple parameters, avoiding raliance on a few weights.
    - Reduces Overfitting:
        - Helps the model generalize better to unseen data by discouraging overly complex representation.
    - Controlled by a Hyperparameter:
        - A regularization coefficient($\lambda$, often set via weight_decay in optimizers) controls the strength of the penalty. Larger values leads to stronger regularization.
    - No Effect on Bias Terms:
        - Regularization is typically applied only to weights, not biases, as biases don't directly affect model complexity.
    - Active During Training:
        -  Regularization affects weight updated only during training. It not explicitly influence the model during inference


In [15]:
class MyANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(32,10)
        )
    
    def forward(self,features):
        return self.model(features)
            

In [16]:
learning_rate = 0.1
epochs = 100

In [ ]:
# instantiating model
model = MyANN(X_train.shape[1])
model = model.to(device)

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay= 1e-4)

In [18]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # forward pass
        outputs = model(batch_features)

        # calculate loss
        loss = criterion(outputs,batch_labels)

        # back pass
        optimizer.zero_grad()
        loss.backward()

        # gradients update
        optimizer.step()

        # total epoch loss
        total_epoch_loss = total_epoch_loss + loss.item()

    # avg loss
    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

Epoch: 1, Loss: 0.7589626661340395
Epoch: 2, Loss: 0.5887992204030355
Epoch: 3, Loss: 0.5483147321343422
Epoch: 4, Loss: 0.5183560692767302
Epoch: 5, Loss: 0.5003774542311827
Epoch: 6, Loss: 0.48681729775667193
Epoch: 7, Loss: 0.4652490175863107
Epoch: 8, Loss: 0.4612256290713946
Epoch: 9, Loss: 0.4486479820609093
Epoch: 10, Loss: 0.4401799097955227
Epoch: 11, Loss: 0.4358231247166793
Epoch: 12, Loss: 0.4259914241532485
Epoch: 13, Loss: 0.4166191813250383
Epoch: 14, Loss: 0.4191155004501343
Epoch: 15, Loss: 0.4068034024288257
Epoch: 16, Loss: 0.40711628329753874
Epoch: 17, Loss: 0.3981456950356563
Epoch: 18, Loss: 0.3987993932242195
Epoch: 19, Loss: 0.39414468891421955
Epoch: 20, Loss: 0.3889881329635779
Epoch: 21, Loss: 0.38558779273927213
Epoch: 22, Loss: 0.38224949114521345
Epoch: 23, Loss: 0.3785603415071964
Epoch: 24, Loss: 0.37915473120907944
Epoch: 25, Loss: 0.3788047790378332
Epoch: 26, Loss: 0.3749391906311115
Epoch: 27, Loss: 0.36984378987550737
Epoch: 28, Loss: 0.36979540838

In [19]:
# set model to eval mode
model.eval()

MyANN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [20]:
# evaluation code for test data
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

    print(correct/total)
    


0.8933333333333333


In [21]:
# evaluation code for training
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in train_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

    print(correct/total)
    

0.9483958333333333


In [33]:
print("Brfore Optimization:",(0.9778541666666667 *100 )- (0.8895 *100))

Brfore Optimization: 8.835416666666674


In [34]:
print("After Optimization:",(0.9483958333333333*100) - (0.8933333333333333 *100))

After Optimization: 5.5062500000000085
